In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import ElementClickInterceptedException
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
# CONFIG
HOTEL_NAME = "Parkroyal Collection Marina Bay Singapore "
PLATFORM = "Agoda"
URL = "https://www.agoda.com/marina-mandarin-singapore-hotel/hotel/singapore-sg.html"
OUTPUT_FILE = "Javian_Raw_Dataset.csv"
MAX_PAGES = 2 # 1 page has 5 reviews, to collect 2000 reviews, we need to scrape 400 pages.

# HELPERS
def human_delay(min_sec=2, max_sec=4):
    time.sleep(random.uniform(min_sec, max_sec))

def scroll_to_bottom(driver):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    human_delay(1,2)

def hide_overlays(driver):
    try:
        driver.execute_script(""" 
            const el = document.querySelector('[data-selenium="checkInBox"]');
            if (el) { el.style.display='none'; }
        """)
    except:
        pass
    human_delay(0.5,1)

# SETUP SELENIUM
options = Options()
options.headless = False
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
driver.get(URL)
human_delay(5,7)

# SCRAPE REVIEWS
reviews_data = []
current_page = 1

while current_page <= MAX_PAGES:
    print(f"Scraping page {current_page}...")

    scroll_to_bottom(driver)
    hide_overlays(driver)
    human_delay(2,3)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    review_cards = soup.select("div.Review-comment")

    if not review_cards:
        print("No reviews found.")
        break

    for r in review_cards:
        try: title = r.select_one("h4[data-testid='review-title']").get_text(strip=True)
        except: title = None
        try: rating = r.select_one("div.Review-comment-leftScore").get_text(strip=True)
        except: rating = None
        try: review_text = r.select_one("p.Review-comment-bodyText").get_text(strip=True)
        except: review_text = None
        try: guest_type = r.select_one("div[data-info-type='group-name'] span").get_text(strip=True)
        except: guest_type = None

        reviews_data.append({
            "title": title,
            "rating": rating,
            "review_text": review_text,
            "guest_type": guest_type
        })

    # Attempt to click next page
    try:
        next_btn = driver.find_element(By.XPATH, f'//button[normalize-space()="{current_page+1}"]')
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_btn)
        human_delay(1,2)
        driver.execute_script("arguments[0].click();", next_btn)
        human_delay(5,7)
        current_page += 1
    except ElementClickInterceptedException:
        print("Click intercepted, retrying...")
        scroll_to_bottom(driver)
        hide_overlays(driver)
        human_delay(2,3)
    except:
        print("Could not click next page. Stopping.")
        break

# SAVE TO CSV
driver.quit()
df = pd.DataFrame(reviews_data)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"Scraping complete. Total reviews: {len(df)}")
print(df.head(5))


Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Scraping page 11...
Scraping page 12...
Scraping page 13...
Scraping page 14...
Scraping page 15...
Scraping page 16...
Scraping page 17...
Scraping page 18...
Scraping page 19...
Scraping page 20...
Scraping page 21...
Scraping page 22...
Scraping page 23...
Scraping page 24...
Scraping page 25...
Scraping page 26...
Scraping page 27...
Scraping page 28...
Scraping page 29...
Scraping page 30...
Scraping page 31...
Scraping page 32...
Scraping page 33...
Scraping page 34...
Scraping page 35...
Scraping page 36...
Scraping page 37...
Scraping page 38...
Scraping page 39...
Scraping page 40...
Scraping page 41...
Scraping page 42...
Scraping page 43...
Scraping page 44...
Scraping page 45...
Scraping page 46...
Scraping page 47...
Scraping page 48...
Scraping page 49...
Scraping page 50...
Scraping 